<a href="https://colab.research.google.com/github/v6mp1r3/workshop/blob/main/ghost_hunt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👻 The AI Ghost Hunt
**SigmoidAI · 31 October 2026 · a warm-up for LMML v2**

The Halloween party starts in 90 minutes. Glitch has stolen all the treats and locked the ghost trap with a password.
The detector at the door keeps calling him a pumpkin.

**Your mission:** fix the detector, follow Glitch's trail to the password, and catch him before the party starts.

### Before you start
1. Choose **File → Save a copy in Drive**. Work in the new tab titled **Copy of…** and close the original.
2. In your copy, run the first code cell with **Shift + Enter** or ▶. Choose **Run anyway** if Colab asks.
3. When you see **READY**, wait for the workshop to start.

Run cells from top to bottom. **Stuck? Raise your hand.**

- ✍️ Fill in a box or finish a line of code. Open the 🔑 cell below an exercise and copy its answer into the ✍️ line if needed.
- 🛑 Stop for a room discussion. Open the section labelled **After the discussion** when the room is ready.
- ⭐ Optional challenges are in the appendix.

<details><summary>Using Jupyter instead of Colab?</summary>

Colab displays some cells as forms. In Jupyter, edit the values beside `#@param` in their code instead.
Use the table of contents to reach the appendix; the jump links are for Colab.

</details>

In [ ]:
#@title ▶ Run me first: unpack the ghost-hunting kit
DATA_URL = "https://drive.google.com/uc?id=1K9sj7dz_KOUt-bdM5wvc9nq6EqCWiRlP"  # the organizers' link to ghost_hunt_data.zip
DATA_SHA256 = "ecc908ffbfaf7a291c60df20f608c4b348004ed53fbadc4b1631f832d3a6fc96"

import hashlib, importlib.util, shutil, subprocess, sys
from pathlib import Path

# Colab has gdown (Drive downloads) and tqdm (Part 2's progress bar) but not pytesseract; install whatever is missing
missing = [pin for pin in ["gdown==5.2.2", "pytesseract==0.3.13", "tqdm==4.67.1"]
           if importlib.util.find_spec(pin.split("==")[0]) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], capture_output=True)
    importlib.invalidate_caches()
if shutil.which("tesseract") is None and "google.colab" in sys.modules:   # Part 2 reads text with Tesseract
    print("Installing Tesseract, the text reader for Part 2 (about 20 seconds)...")
    install = "apt-get -qq install -y tesseract-ocr"   # if the package lists are stale, refresh them and try again
    subprocess.run(f"{install} || (apt-get -qq update && {install})", shell=True, capture_output=True)

kit = Path("ghost_hunt").resolve()
if not (kit / "ghosthunt.py").exists():
    archive = Path("ghost_hunt_data.zip")
    ask_a_helper = ("Ask a helper for ghost_hunt_data.zip, upload it with the Files panel on the left, then run this "
                    "cell again.")
    if not archive.exists():
        if not DATA_URL:
            raise RuntimeError("No download link yet. " + ask_a_helper)
        print("Downloading the kit (about 12 MB)...")
        import gdown
        try:
            gdown.download(DATA_URL, str(archive), quiet=True)
        except Exception as error:  # Drive refused it, or the network dropped.
            archive.unlink(missing_ok=True)
            raise RuntimeError(f"The download failed ({type(error).__name__}). " + ask_a_helper) from None
        if not archive.exists():
            raise RuntimeError("The download failed. " + ask_a_helper)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != DATA_SHA256:
        archive.unlink()
        raise RuntimeError("The kit didn't arrive in one piece. Run this cell again; if it fails twice, ask a helper.")
    shutil.unpack_archive(archive)
if str(kit) not in sys.path:
    sys.path.insert(0, str(kit))

import ghosthunt as gh
import ghost_tools as tools
tools.startup_check()
if shutil.which("tesseract") is None:
    tools.warning("Tesseract didn't install, so Part 2 can't read the timestamps itself. Run this cell again; if it still "
                  "fails, Part 2 has a rescue cell with the readings.")
tools.note("You're set if this notebook's title starts with <b>Copy of</b>. If it doesn't, make your copy now "
           "(<b>File → Save a copy in Drive</b>) and run this cell again there. Then wait here until the workshop starts. "
           "Don't read ahead: it spoils the puzzles we'll solve together.")

## Today's mission

| Part | Your task | LMML continent |
|---|---|---|
| **1 · Train** | Find out what the detector learned, and fix it | Asia · CV |
| **2 · Track** | Follow Glitch through the cameras to find the password | Europe · EDA |
| **3 · Trick** | Make the smallest disguise that fools the exit camera | South America · security in AI |

---
# Part 1 · Train: what is the detector looking at?

The security team trained the door camera's detector on pictures they took. The cells below are their starter
notebook: they load those pictures, train a detector and measure its accuracy.
Some LMML tasks come with a starter notebook like this one. When a task doesn't, these three steps are the first code
you write. The detector is a **model** that learns from examples. What rule will it learn?

### The security team's pictures

In `ghost_hunt/part1_train/`:
- `train/` holds 600 **training pictures**, the examples the model learns from.
- `train_labels.csv` gives each picture's **label**: ghost or pumpkin.
- `validation/` holds 100 pictures kept aside for checking, with labels in `validation_labels.csv`.

Open files in Colab's Files panel. `ghost_hunt/README.md` describes the full dataset.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

PART1 = Path("ghost_hunt/part1_train")

def load_pictures(split):
    table = pd.read_csv(PART1 / f"{split}_labels.csv")
    paths = [PART1 / split / name for name in table["file"]]
    images = np.stack([np.array(Image.open(path)) for path in paths])
    label_ids = table["label"].map({"ghost": 0, "pumpkin": 1}).to_numpy()
    return {"images": images, "labels": label_ids}

train = load_pictures("train")
validation = load_pictures("validation")
print(train["images"].shape, validation["images"].shape)  # Images, rows, columns, colour channels.
pd.read_csv(PART1 / "train_labels.csv")["label"].value_counts()

### Train the model

We use **PyTorch** to build and train the detector. Its **layers** combine simple patterns, such as edges and colours,
into more complex ones. The final layer gives a score for each class: ghost and pumpkin.

In [ ]:
import torch
from torch import nn

def new_model():
    torch.manual_seed(0)  # Same starting weights on every run.
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 2),
    )

The **loss** measures how wrong the predictions are. The **optimizer** adjusts the model's weights to reduce it.
`loss.backward()` computes **gradients**, which tell the optimizer how to adjust each weight.
Training repeats this process in batches of 32 pictures, making eight passes through the training set.

In [ ]:
def train_model(images, labels, passes=8):
    model = new_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    batches = gh.Batches(images, labels)  # Shuffled batches of 32 images.

    for pass_index in range(passes):
        total_loss = 0
        for batch_images, batch_labels in batches:
            # Predict and measure the error.
            batch_scores = model(batch_images)
            loss = nn.functional.cross_entropy(batch_scores, batch_labels)

            # Update the weights.
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch_images)

        average_loss = total_loss / len(labels)
        print(f"pass {pass_index + 1} of {passes}: average loss {average_loss:.3f}")
    return model

detector = train_model(train["images"], train["labels"])

### Check it on new pictures, then on Glitch

How quickly did the training loss fall?

The 100 **validation pictures** were taken the same way as the training pictures, but the model hasn't trained on them.
Check its accuracy, then look at its answer for Glitch from the door camera.

Each picture shows the predicted class and a **prediction score**: the model's confidence, which can be high even when
it's wrong.

In [ ]:
def scores(model, images):
    # Convert images to tensors; return [ghost, pumpkin] probabilities that sum to 1.
    with torch.no_grad():
        model_scores = model(gh.to_tensor(images))
        return model_scores.softmax(dim=1).numpy()

def predict(model, images):
    class_ids = scores(model, images).argmax(axis=1)
    return np.array(gh.CLASSES)[class_ids]

glitch = np.array(Image.open(PART1 / "door_camera.png"))
for name, data in [("training", train), ("validation", validation)]:
    expected_labels = np.array(gh.CLASSES)[data["labels"]]  # Class IDs to names.
    predictions = predict(detector, data["images"])
    accuracy = (predictions == expected_labels).mean()
    print(f"It labels {accuracy:.0%} of the {len(expected_labels)} {name} pictures correctly.")

preview_images = np.concatenate([validation["images"][8:15], glitch[None]])  # Add Glitch as the last image.
preview_labels = np.append(validation["labels"][8:15], 0)  # Glitch's label: ghost = 0.
tools.show_answers(detector, preview_images, preview_labels)
tools.glitch("Boo. I mean... pumpkin.")

### What did it learn from?

Training and validation scores look good. Why does it misclassify Glitch? Inspect the training pictures:
1. **Are the labels right?** Check these first.
2. **If so, what else differs between ghosts and pumpkins?** Consider the whole picture.

Write your observations below. We'll return to them after testing.

In [ ]:
captions = [gh.CLASSES[label_id] for label_id in train["labels"]]
tools.show_pictures(train["images"], captions, count=16, seed=839)

In [ ]:
#@title ✍️ What's different between the ghost pictures and the pumpkin pictures?
differences = ""  #@param {type:"string"}

### Hide a region, watch the answer

Cover a small square and measure how much the model's confidence drops. Repeat across the image to make an
**occlusion map**. **Red marks regions where covering the image weakened its prediction.**

Read the loop and complete the ✍️ line: calculate the drop from `before` to `after`.
Before running it, predict where the red will appear: on Glitch or around him?

In [ ]:
def confidence(model, image, answer):
    with torch.no_grad():
        ghost_score, pumpkin_score = model(gh.to_tensor(image[None]))[0]  # [None] adds a batch dimension.
    # Use the gap between raw scores, before converting to probabilities.
    if answer == "ghost":
        return float(ghost_score - pumpkin_score)
    return float(pumpkin_score - ghost_score)

def occlusion_map(model, image):
    answer = predict(model, image[None])[0]
    before = confidence(model, image, answer)
    grey = np.median(image)  # Neutral cover based on this image's pixel values.
    heatmap = np.zeros((64, 64))

    # Cover one 8 x 8 square at a time.
    for top in range(0, 64, 8):
        for left in range(0, 64, 8):
            covered = image.copy()
            covered[top:top + 8, left:left + 8] = grey
            after = confidence(model, covered, answer)
            heatmap[top:top + 8, left:left + 8] = 0  # ✍️ Replace 0 with the drop in confidence.

    tools.show_map(image, heatmap, f"says {answer}", model=model)

occlusion_map(detector, glitch)

In [ ]:
#@title 🔑 Show answer
# heatmap[top:top + 8, left:left + 8] = before - after

### What about a correct prediction?

Make the same map for the daylight ghost from the validation grid. The detector classified this one correctly.

In [ ]:
right_ghost = validation["images"][8]  # Daylight ghost from the validation grid.
occlusion_map(detector, right_ghost)

**Which regions affect the predictions most: the ghosts or their backgrounds?** Compare both maps.

<details><summary>What the maps suggest</summary>

Covering the backgrounds weakens both predictions; covering the ghosts has little effect. The detector may be using
the background to decide, even when its answer is correct.

</details>

**The map is a clue, not proof.** Test the suspected feature by changing it.

### Change one thing and test your guess

Start with the daylight ghost. Each pixel has red, green and blue channels, numbered 0, 1 and 2.
`images[..., order]` rearranges them; `[0, 1, 2]` leaves them unchanged.

Swapping red and blue exchanges orange and blue light. The white ghost, black outline and day/night setting stay the same.

✍️ **Change the channel order in `swap_light` to swap red and blue.** Predict whether the model's answer will change,
then test it.

In [ ]:
def swap_light(images):
    # ... keeps the other axes, so this works on one image or a batch.
    return images[..., [0, 1, 2]]  # ✍️ Swap red (0) and blue (2); keep green (1) in place.

image = right_ghost
swapped_image = swap_light(image)
before, after = predict(detector, np.stack([image, swapped_image]))
tools.show_before_after([image], [swapped_image], [before], [after])

In [ ]:
#@title 🔑 Show answer
#     return images[..., [2, 1, 0]]

Does the result hold across more pictures? Swap the light in all 600 training images and count how many predictions
change. The grid shows eight examples; a red ✗ marks a changed answer.

In [ ]:
swapped_images = swap_light(train["images"])
before = predict(detector, train["images"])
after = predict(detector, swapped_images)
changed_count = (before != after).sum()
print(f"Changing only the colour of the light flipped {changed_count} of {len(before)} answers.")

tools.show_before_after(train["images"], swapped_images, before, after)
tools.reveal(detector, globals().get("differences", ""))

🛑 **The detector learned to read the colour of the light. How would you unteach it that rule?** Answer out loud.

## After the discussion: try a repair

**Fix it with counterexamples**

Add ghosts in moonlight and pumpkins in lamplight. Your colour swap creates these missing examples from the security
team's pictures.

Creating new training examples from existing ones is **data augmentation**. The change must preserve the label:
a ghost in different light is still a ghost. Colour changes would be risky for classifying ripe and unripe bananas.

✍️ **Create a swapped copy of every training picture, then give each copy its original label.** Complete both lines.

In [ ]:
copies = train["images"]  # ✍️ Make a swapped copy of every training image.
copy_labels = []          # ✍️ Give each copy its label.
all_images = np.concatenate([train["images"], copies])
all_labels = np.concatenate([train["labels"], copy_labels])
print(f"{len(train['images'])} training pictures + {len(copies)} swapped copies = {len(all_images)} pictures")
copies_ok = tools.check_copies(copies, copy_labels)

In [ ]:
#@title 🔑 Show answer
# copies = swap_light(train["images"])
# copy_labels = train["labels"]  # Changing the light preserves the label.

### Retrain and check the repair

Will the loss fall as quickly this time? Retrain, then compare Glitch's prediction and occlusion map before and after
the repair.

In [ ]:
if not copies_ok:
    print("Your copies aren't ready yet. Fix the two ✍️ lines above, run that cell, then run this one again.")
else:
    print(f"Retraining with your {len(copies)} swapped copies...")
    repaired_detector = train_model(all_images, all_labels, passes=12)
    print("Before the repair:")
    occlusion_map(detector, glitch)
    print("After the repair:")
    occlusion_map(repaired_detector, glitch)
    tools.glitch("Hey! Stop looking at my face.")

### Test both detectors on the corridor

The corridor cameras provide 40 **test pictures** that neither model has trained on. Unlike the security team's
pictures, these include ghosts in moonlight and pumpkins under lamps. Compare accuracy for each group.

The pictures are in `ghost_hunt/test/`. Their labels are in `ghost_hunt/grader/answers/test_labels.csv`, which we use
only for scoring. At LMML, the grader keeps test labels hidden and returns your score.

In [ ]:
paths = sorted(Path("ghost_hunt/test").glob("*.png"))
test_images = np.stack([np.array(Image.open(path)) for path in paths])
answers = pd.read_csv("ghost_hunt/grader/answers/test_labels.csv")
image_ids = [path.stem for path in paths]
results = answers.set_index("id").loc[image_ids]  # Match labels to image order.

for name, model in [("original", detector), ("repaired", repaired_detector)]:
    predictions = predict(model, test_images)
    results[name] = predictions == results["label"]

tools.note("Some of these ghosts were hiding in plain sight. Now we can follow Glitch.")
group_accuracy = results.groupby(["label", "light"])[["original", "repaired"]].mean()
group_accuracy.map(lambda accuracy: f"{accuracy:.0%} right")

## 🛑 Let's discuss

**What might a new camera show that our training pictures don't cover?** Suggest some possibilities. Answer out loud.

## After the discussion: ideas for new cameras

- **Position and size:** smaller, larger, off-centre, cut off at the edge
- **Angle:** mirrored, tilted, sideways, upside down
- **Focus:** blurry, zoomed in
- **Light and colour:** darker, brighter, black and white, another colour of light (the one you just fixed)
- **Overlays:** grainy pictures, text on the frame (Part 2 comes back to this)

Any of these can become training copies, as long as the label stays true.
⭐ [Test your ideas in the optional challenge](#scrollTo=extra-intro).

Continue to Part 2 when the room moves on.

---
# Part 2 · Track: where is Glitch now?

Twenty cameras line the corridor, one per metre from −6 m to +13 m. Each recorded one frame per second for
12 seconds. Other ghosts were there too.

**Your goal:** find Glitch's route, the camera that saw him at each second from 1 to 12.
Those 12 letters are the password that unlocks the ghost trap.

| Brief | Details |
|---|---|
| **Data** | 240 PNGs in `ghost_hunt/part2_track/cameras/`, in shuffled order |
| **Filenames** | `037_R_+3.png` means arrival 037, camera R, position +3 m: where, not when. The timestamp in each frame's top-left corner says when. |
| **Rules** | Glitch starts at 0 m at 21:40:00 and moves at most 3 m per second. Exactly one route reaches second 12. |
| **Submit** | The 12 letters in order, in the exit console |

**Plan:** read each frame's time → find the frames that show a ghost → follow the trail from 0 m.

In [ ]:
#@title 🆘 Check or replace the detector
use_reference_detector = False  #@param {type:"boolean"}
import numpy as np
if "predict" not in globals() or "occlusion_map" not in globals():   # Restore helpers if Part 1 was skipped.
    def predict(model, images):
        return np.array(gh.predict(model, images))
    def occlusion_map(model, image):
        tools.show_occlusion(model, "Hiding each square", image, "this frame")
if use_reference_detector or "repaired_detector" not in globals():
    repaired_detector = tools.load_detector("repaired")
    print("Part 2 now uses the reference repaired detector, retrained with swapped copies of both ghosts and pumpkins.")
elif gaps := tools.exam_gaps(repaired_detector):
    tools.warning(f"🎃 <b>Your repaired detector misses part of the corridor's test pictures:</b> {'; '.join(gaps)}. "
                  "Tick the box and run this cell again to use the reference repaired detector.")
else:
    print("Your repaired detector holds up on the corridor's test pictures, so Part 2 uses it. Nothing to rescue.")

### Read the timestamps with OCR

Inspect frame 0 and the first eight frames. Where is the timestamp? What colour are its text and background?
How large is it? The axes count pixel rows and columns.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

paths = sorted(Path("ghost_hunt/part2_track/cameras").glob("*.png"))
frames = np.stack([np.array(Image.open(path)) for path in paths])
print(frames.shape)  # Frames, rows, columns, colour channels.
plt.figure(figsize=(4, 4))
plt.imshow(frames[0])  # Axes show pixel coordinates.
plt.title(paths[0].name)
plt.show()

In [ ]:
tools.show_pictures(frames[:8], [path.name for path in paths[:8]], count=8, columns=4, size=2.2)

**OCR** (optical character recognition) reads text from images. We'll use **Tesseract**, an OCR engine.
The timestamp is small white text on black.

Before running it, predict: will Tesseract find the timestamp in the whole frame?
An empty reading (`''`) means Tesseract found no text.

In [ ]:
import pandas as pd
import pytesseract
from IPython.display import display

whole = pytesseract.image_to_string(Image.fromarray(frames[0])).strip()
print(f"whole frame: '{whole}'")

✍️ **Crop to the black box.** `frame[:rows, :columns]` keeps the top rows and the first columns. Find the box's size
on frame 0's picture above: the left axis counts rows, the bottom axis counts columns.

In [ ]:
box_rows, box_columns = 128, 128  # ✍️ How many rows and columns does the black box fill?
crop = Image.fromarray(frames[0][:box_rows, :box_columns])  # Top rows, first columns.
display(crop)
crop_ok = tools.check_crop(box_rows, box_columns)

In [ ]:
#@title 🔑 Show answer
# box_rows, box_columns = 20, 90

With only the box, will Tesseract read the time?

In [ ]:
reading = pytesseract.image_to_string(crop).strip()
print(f"crop: '{reading}'")

Tesseract was built for scanned printed pages: dark text on white paper, letters large enough to read clearly, in
lines and paragraphs. How is this crop different?

Make it look more like a page: dark text on white, enlarged three times. Then tell Tesseract what to expect:
`--psm 7` reads a single line, and the whitelist allows only digits and colons.

In [ ]:
def clean_timestamp(frame):
    box = 255 - frame[:box_rows, :box_columns, 0]  # Crop and invert to dark text on white.
    box = Image.fromarray(box)
    return box.resize((box_columns * 3, box_rows * 3), Image.Resampling.BICUBIC)

ocr_config = "--psm 7 -c tessedit_char_whitelist=0123456789:"  # One line; digits and colons only.

def ocr(frame):
    text = pytesseract.image_to_string(clean_timestamp(frame), config=ocr_config)
    return text.strip()

display(clean_timestamp(frames[0]))
print(f"clean timestamp: '{ocr(frames[0])}'")

Compare whole-frame readings and cleaned timestamps across 16 frames. In how many frames does each column read a full
time, like `21:40:07`?

In [ ]:
rows = []
for frame in frames[:16]:
    whole = pytesseract.image_to_string(Image.fromarray(frame)).strip()
    cleaned = ocr(frame)
    rows.append({"whole frame": f"'{whole}'", "clean timestamp": f"'{cleaned}'"})

display(pd.DataFrame(rows).rename_axis("frame"))

Tesseract reads the time in only some whole frames, but in all 16 clean timestamps.

Three OCR habits:
1. **Give it a clean input.** Crop to the text, make it dark on white and enlarge small characters.
2. **Tell it what to expect.** Here: one line of digits and colons.
3. **Check every reading.** Reject readings that aren't a real time. Passing the check still doesn't prove the time
   is right.

🛑 **The next cells build a table with each frame's camera, position and second. What values should each column hold?
Which would be impossible?** Run the next cell to read all 240 timestamps; it takes about a minute. Answer out loud
while it runs.

In [ ]:
from tqdm.auto import tqdm  # Progress bar for the loop.

texts = [ocr(frame) for frame in tqdm(frames, desc="Reading timestamps")]
texts[:5]

In [ ]:
import re

RECORDING_START = 21 * 3600 + 40 * 60  # 21:40:00, in seconds after midnight.

# "21:40:07" becomes 7, the seconds after 21:40:00. None if it isn't a real time.
def read_timestamp(text):
    if not re.fullmatch(r"\d\d:\d\d:\d\d", text):
        return None  # Not in the format 21:40:07.
    hours, minutes, seconds = [int(part) for part in text.split(":")]
    if hours > 23 or minutes > 59 or seconds > 59:
        return None  # An impossible time, like 21:40:67.
    return hours * 3600 + minutes * 60 + seconds - RECORDING_START

readings = [read_timestamp(text) for text in texts]
failed = readings.count(None)
print(f"{len(readings) - failed} of {len(readings)} readings passed the check.")
if failed:
    first = readings.index(None)
    tools.warning(f"{failed} of {len(readings)} readings failed the check, so they aren't used. For example, frame "
                  f"{first} read '{texts[first]}'. Tick the rescue box below and run it.")

**Spot-check the readings.** The check only tests the format. Does each caption match the timestamp in its picture?

In [ ]:
captions = [f"{i}: '{text}'" for i, text in enumerate(texts)]
tools.show_pictures(frames, captions, count=16, columns=8, seed=2)

In [ ]:
names = [path.stem.split("_") for path in paths]  # 037_R_+3 -> arrival, camera, position.
camera_log = pd.DataFrame({
    "camera": [camera for _, camera, _ in names],
    "position": [int(position) for _, _, position in names],
    "second": readings,
})
camera_log = camera_log.dropna().astype({"second": int})  # Discard failed readings; preserve frame indices.
camera_log.head()

In [ ]:
#@title 🆘 Load saved readings if OCR failed
use_saved_readings = False  #@param {type:"boolean"}
if use_saved_readings or "camera_log" not in globals():
    frames = gh.load_frames()
    camera_log = tools.saved_readings()
    print(f"Loaded the saved readings of all {len(camera_log)} timestamps.")
else:
    print(f"You have your own readings of {len(camera_log)} timestamps, so nothing to rescue.")

### Check the table before using it

The 🛑 question above asked what values each column should hold. Compare the summary with your answer:
`min` and `max` give each column's range.

In [ ]:
camera_log.describe()

If a column looks wrong, find which camera it comes from.

In [ ]:
camera_log.groupby("camera")["second"].agg(["min", "max"])

Did OCR misread them? Look at the timestamps.

In [ ]:
late_frames = camera_log[camera_log["second"] > 12].index
tools.show_pictures(frames[late_frames], [f"{i}: second {camera_log['second'][i]}" for i in late_frames],
                    count=12, columns=6)

**The timestamps really say 22:40. Why would one camera's clock be an hour ahead?** Discuss before opening the answer.

<details><summary>🔑 Open the answer</summary>

One camera is an hour ahead. It didn't adjust when summer time ended on Sunday, 25 October.
Enter its letter below to subtract an hour from its timestamps.

</details>

In [ ]:
#@title ✍️ Fix the time
camera = ""  #@param {type:"string"}
camera = camera.strip().upper()
late = (camera_log["camera"] == camera) & (camera_log["second"] > 3600)
camera_log.loc[late, "second"] -= 3600  # Subtract one hour; leave already-corrected rows alone.

if (camera_log["second"] > 12).any():
    tools.warning("Some frames still have timestamps after 21:40:12. Which camera do they come from?")
else:
    tools.glitch(f"Summer's over, camera {camera}.")

🛑 **What will your repaired detector say about frames with timestamps in the corner?** Answer out loud, then run the
next cell.

In [ ]:
# The detector was trained on 64 x 64 pictures: keep every other pixel of each 128 x 128 frame.
small_frames = frames[:, ::2, ::2]
camera_log["prediction"] = predict(repaired_detector, small_frames[camera_log.index])  # Only frames with a reading.
ghosts = camera_log[camera_log["prediction"] == "ghost"]
print(f"Your repaired detector sees {len(ghosts)} ghosts in {len(camera_log)} frames.")

### Where did the ghosts go?

Find a ghost the detector calls a pumpkin in the 16 frames below. Then ask Part 1's question again: what is the
detector looking at? Enter that frame's number below.

In [ ]:
captions = [f"{i}: {camera_log['prediction'].get(i, 'not read')}" for i in range(16)]
tools.show_pictures(small_frames, captions, count=16)

In [ ]:
#@title ✍️ Inspect a misclassified ghost
frame_number = -1  #@param {type:"integer"}
if frame_number < 0:
    print("Type the number of a frame showing a ghost the detector calls a pumpkin.")
else:
    occlusion_map(repaired_detector, small_frames[frame_number])

**Which region affects the prediction most?**

<details><summary>A clue from the map</summary>

The timestamp affects the prediction. The training pictures had no text in the corner.

</details>

Test this by covering the timestamp with the frame's bottom-left corner: wall in the same light.
This keeps the ghost visible, though a low ghost may leave a small trace in the copied patch.

In [ ]:
clean_frames = small_frames.copy()
for frame in clean_frames:
    frame[:11, :46] = frame[-11:, :46]  # Cover the timestamp with the bottom-left wall.

camera_log["prediction"] = predict(repaired_detector, clean_frames[camera_log.index])
ghosts = camera_log[camera_log["prediction"] == "ghost"]
print(f"Your repaired detector now sees {len(ghosts)} ghosts in {len(camera_log)} frames.")
captions = [f"{i}: {camera_log['prediction'].get(i, 'not read')}" for i in range(16)]
tools.show_pictures(clean_frames, captions, count=16)
tools.glitch("Hey! Who let all those ghosts back in?")

Compare the ghost count before and after covering the timestamps. Even the repaired model was sensitive to a detail
absent from its training data.

Use consistent image preparation during training and testing. Now plot the ghost detections.

In [ ]:
tools.plot_ghost_positions(ghosts)

In [ ]:
#@title 🆘 Load the reference table to catch up
replace_my_table = False  #@param {type:"boolean"}
if replace_my_table or "ghosts" not in globals():
    camera_log, frames, clean_frames = tools.reference_table()
    ghosts = camera_log[camera_log["prediction"] == "ghost"]
    print("Loaded the reference table: timestamps read, timestamps covered, the reference repair's answers, clock fixed.")
else:
    ghosts = camera_log[camera_log["prediction"] == "ghost"]  # Picks up a clock fixed after the timestamp was covered.
    if (ghosts.second > 12).any():
        print("Some of your ghosts are still after 21:40:12: fix the clock in ✍️ Fix the time, then run this cell again.")
    else:
        print("You already have your own table, so the plot below uses yours. Nothing to rescue.")

### Follow Glitch

Each dot is a ghost detection labelled with its camera letter. Grey lines connect possible steps: one second later
and at most 3 m away.

Start at the star and follow the lines to second 12. Dead ends and disconnected ghosts are distractions.
Exactly one route reaches the end. Enter its 12 letters below.

In [ ]:
tools.plot_ghost_positions(ghosts, links=True)

In [ ]:
#@title 🔐 Enter the 12 camera letters
code = ""  #@param {type:"string"}
if tools.unlock(code):
    tools.show_route(ghosts, frames, code)

⭐ [Optional: write a program that finds the route](#scrollTo=track-extra-intro).
Continue to Part 3 when the room moves on.

---
# Part 3 · Trick: the bait

Glitch needs a disguise before he'll enter the trap. Make the exit camera classify him as a pumpkin.
It uses a fixed repaired detector, trained like yours with swapped copies of both classes.

**Ask first:** What does success look like? What tools do I have? What's my first experiment? Ask these of any
challenge, today and at LMML.

**Paint by hand, only inside the dashed halo.** Keep the ghost's outline and everything outside the halo unchanged.
Don't change the model.

Start with **Test clean image**, then paint and use **Test pattern**. Watch the pumpkin score as you experiment.

**Check submission** validates your disguise. Among successful disguises, the fewest painted cells wins. Hints are
below the editor.

In [ ]:
editor = tools.MaskEditor()
editor

<details><summary>Prefer typing to clicking?</summary>

Use `editor.show_numbers()` to see cell numbers, then `editor.paint([3, 4, 5], "white")`,
`editor.test_pattern()` and `editor.check()`.
To test on your Part 1 model, use `editor.test_pattern(repaired_detector, "your detector")`.

</details>

In [ ]:
#@title 💡 Choose a hint
hint_number = 1  #@param [1, 2, 3] {type:"raw"}
tools.hint(hint_number)

⭐ [Optional: write a program that paints a disguise](#scrollTo=trick-attack-intro).
Return for the finale when the room is ready.

---
## 🪤 The trap

Eyes on the big screen: Glitch is about to try his disguise at the exit. When the scene ends, run the two cells below
to replay it with the disguise in your editor.

In [ ]:
#@title 🎭 Glitch takes the disguise in the editor
exit_camera = tools.load_detector("repaired")
if "editor" not in globals():
    print("Run the editor cell in Part 3 first.")
else:
    exit_result = tools.glitch_exit(editor.code, exit_camera)

In [ ]:
#@title 🪤 Close the trap
if "exit_result" in globals() and exit_result["says"] == "pumpkin":
    tools.close_trap(editor.code, exit_camera)
else:
    print("Glitch only steps into the doorway in a disguise the exit camera calls a pumpkin. Run the cell above first.")

---
# Your kit for LMML v2

In two weeks: 24 hours and a world map of ML tasks. Take these habits with you:

1. Read the submission format, rules and scoring before starting.
2. Get a simple first version working end to end, then improve it.
3. Inspect the data and labels. Investigate scores that look too good.
4. Change one thing at a time and record the result.
5. Test on unseen data and check performance across different groups.

[LMML reference: task briefs, AI assistants and useful tools](#scrollTo=lmml-reference).

👻 **Glitch:** *"You caught me this time. I've hidden more flags in LMML. See you there."*

The main workshop ends here. The challenges below are optional.

---
# Appendix: optional challenges

Choose a challenge after completing its workshop part. Each uses variables from that part and has a return link.

## ⭐ Part 1 extra: improve the detector

**Start here after completing Part 1.** [Return to Part 2](#scrollTo=track-intro) when the room moves on.

`ghost_hunt/part1_train/challenge/` holds 144 pictures from unfamiliar cameras. Improve accuracy on them:

1. Score `repaired_detector` and inspect its mistakes. What do they share?
2. Fill in `changes` to augment your 1,200 training images. `apply_to_images(images, transform)` applies a
   transformation of one PIL image to the whole batch. Each change must preserve the label.
3. Train `best_detector`, score it and inspect the remaining mistakes. More copies take longer to train.

When satisfied, check the **final set**: unseen pictures of the same kinds. A much lower score suggests your changes
fit the challenge examples too closely.

This exercise creates `best_detector`; it leaves `repaired_detector` unchanged.

<details><summary>💡 Hint: some PIL tools</summary>

`image.rotate(angle)`, `image.filter(ImageFilter.GaussianBlur(radius))`, `image.resize(size)` and
`ImageEnhance.Brightness(image).enhance(factor)`. Which of them matches the mistakes you see is up to you.

</details>

In [ ]:
tools.score_challenge(repaired_detector)

In [ ]:
from PIL import ImageEnhance, ImageFilter

def apply_to_images(images, transform):
    changed_images = []
    for image in images:
        changed_image = transform(Image.fromarray(image)).convert("RGB")
        changed_images.append(np.array(changed_image))
    return np.stack(changed_images)

In [ ]:
changes = [  # Transformations of one PIL image, e.g. lambda image: image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
]
copies = [apply_to_images(all_images, transform) for transform in changes]
augmented_images = np.concatenate([all_images, *copies])
augmented_labels = np.concatenate([all_labels] * (len(copies) + 1))  # Each copy keeps its label.
copy_count = sum(len(batch) for batch in copies)
print(f"{len(all_images)} pictures from your repair + {copy_count} changed copies; retraining...")

best_detector = train_model(augmented_images, augmented_labels, passes=12)
tools.score_challenge(best_detector)

In [ ]:
if "best_detector" in globals():
    tools.score_final(best_detector)
else:
    print("Train your detector in the cell above first.")

## ⭐ Part 2 extra: find the route with code

**Complete Part 2 first.** [Return to Part 3](#scrollTo=trick-card) when the room moves on.

Write `find_route(ghosts)` using any method. It receives a table of ghost sightings with `camera`, `position` and
`second` columns, in any order, and a sighting can appear twice. It returns Glitch's camera letters for seconds 1 to 12
as one string.

- Glitch starts at 0 m at 21:40:00 (second 0). The start is not in the table.
- Each second, he steps to a sighting from the next second, at most 3 m away: left, right or the same spot.
- Exactly one route reaches second 12; other trails stop earlier.

The checker runs your function on this corridor and on 3 other nights in the same corridor that you never see. It
gives each 10 seconds and says only right or wrong.

<details><summary>💡 Hint 1</summary>

Write a filter that finds every possible next ghost from a given position and second.

</details>
<details><summary>💡 Hint 2</summary>

Follow a path until it ends, then backtrack and try another. This is **depth-first search**; a function can call itself
to explore each branch.

</details>
<details><summary>💡 Hint 3</summary>

Use `(ghosts.position - position).abs()` for distance.

</details>

Finished? Undo the clock fix on a copy and try again:
`before = ghosts.copy()`, then `before.loc[before.camera == camera.strip().upper(), "second"] += 3600`.

In [ ]:
def find_route(ghosts):
    return ""  # Your code: return the 12 camera letters.

tools.check_route_finder(find_route)

## ⭐ Part 3 extra: paint with code

**Open the Part 3 editor first.** [Return to the trap](#scrollTo=showdown) for the finale.

Write a program that makes the exit camera classify Glitch as a pumpkin. You may edit individual pixels, so these
disguises don't enter the hand-painted leaderboard.

| Tool | What it provides |
|---|---|
| `editor.target` | Clean image, shape 64 × 64 × 3, values 0–255 |
| `editor.ring` | Boolean mask of allowed halo pixels |
| `editor.detector` | Fixed exit-camera model |
| `gh.scores(model, pictures)` | Scores a batch of images |
| `gh.to_tensor(pictures)` | Converts images for PyTorch |

**Rules:** change only halo pixels; keep each colour channel within `255 * limit` of its original value.
At `limit = 0.3`, that's about 76. Keep the model fixed.
Validate with `tools.check_disguise(picture, limit)`.

Succeeded at 0.3? Try a smaller limit. Our best result was 0.23.

<details><summary>💡 Hint 1: search</summary>

Change one pixel's colour within the limit. Keep changes that raise the pumpkin score.
Score many candidates together with `gh.scores` to save time.

</details>
<details><summary>💡 Hint 2: gradients</summary>

Training adjusted weights using `loss.backward()`. Keep the weights fixed and adjust the pixels instead.

</details>

In [ ]:
import numpy as np

limit = 0.3  # Maximum channel change: 0.3 * 255 is about 76.
picture = editor.target.copy()
# Your code: change halo pixels only.

tools.check_disguise(picture, limit)

Try your method on `tools.load_detector("door")` too. How do the patterns differ?

<details><summary>What we found with a gradient attack</summary>

The door camera falls for blue light, the shortcut from Part 1. The exit camera takes longer to fool and
produces a fine pattern of light and dark pixels. Test each disguise on both models to check whether it transfers.

The colour repair still leaves other weaknesses. **Adversarial training** adds attacked images to the training set,
much as you added swapped copies.

</details>

---
# LMML reference

### Read the task brief

| Section | Check |
|---|---|
| 🎯 Objective | What must you produce? |
| 📂 Dataset | What does each file and column contain? |
| ⚙️ Requirements | Which Python version and libraries are allowed? |
| ✅ Expected Output | What do you submit, and in which exact format? |
| 🏆 Scoring Criteria | How is it scored? Is partial credit available? |
| 💡 Hints | What does each hint cost? |

For a `main.py` submission, check how it must read input files, including any bare-filename requirement.

### Use an AI assistant to learn faster

LMML is a learning marathon: use your assistant to learn faster, not to skip the learning.

| Use it to | Try asking |
|---|---|
| See your data sooner | "Write code that shows 16 random samples and the label counts." Then do the looking yourself. |
| Learn a method that's new to you | "Show me this method on a 10-row example, then explain each step." |
| Plan experiments you'll learn from | "List five experiments, cheapest first, and what each result would tell me." |
| Understand every line it writes | "Explain this cell line by line, then quiz me on why each step is there." |
| Get unstuck on an error | "Why does this error happen?" before "Fix it." |
| Test before you submit | "Write tests that would catch my mistakes: file name, output format, edge cases." Read each test before you trust it. |

**Your assistant helps; your team answers for it.** Be ready to explain any part of your solution. If you can't,
the team is penalized.

### Useful tools

**Set up before you arrive.** Use one environment per task, because each brief pins its own versions. Free GPUs:
Colab, Kaggle (verify your phone number on Kaggle to unlock its GPUs).

| Continent | Good starting points |
|---|---|
| Europe · EDA | pandas, scikit-learn, Matplotlib, SHAP |
| Asia · CV | Pillow, OpenCV, torchvision |
| North America · NLP | Hugging Face transformers and datasets, spaCy |
| Africa · General AI | Generative AI, Audio and Agents, SLMs |
| South America · security in AI | PyTorch, NumPy, SciPy |
| Australia · RL | Gymnasium: environments with rewards, for training agents and their policies |

**Start from a pretrained model** on [Hugging Face](https://huggingface.co/models):
1. **Find:** filter by task and language, then sort by downloads. Small models are quicker to try.
2. **Check:** read the model card: what it was trained for, its size, and where it fails.
3. **Load:** copy the ID or the "Use this model" code, and pin a revision so results don't change.

Documentation: [Colab](https://research.google.com/colaboratory/faq.html) ·
[Jupyter](https://docs.jupyter.org/en/latest/) ·
[pandas](https://pandas.pydata.org/docs/getting_started/intro_tutorials/02_read_write.html) ·
[Matplotlib](https://matplotlib.org/stable/tutorials/pyplot.html) ·
[scikit-learn](https://scikit-learn.org/stable/getting_started.html) ·
[SHAP](https://shap.readthedocs.io/en/latest/) ·
[Hugging Face Hub](https://huggingface.co/docs/hub/en/index).

[Back to the LMML checklist](#scrollTo=lmml-kit).